In [9]:
%pip install -q beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install sentence-transformers

# Configure the .env

In [1]:
import os
from dotenv import load_dotenv
from getpass import getpass

try:
    load_dotenv()
except Exception as e:
    pass

if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter your TAVILY API key: ")
    
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter your PINECONE API key: ")
    
if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API key: ")
    
    
print("GROQ_API_KEY configured:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))


GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


#LOADING WEB PAGE

In [2]:
from langchain_community.document_loaders import WebBaseLoader

source = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_path=source,
    requests_kwargs={
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)

docs = loader.load()
print("Documents loaded: ",len(docs))
print("metadata: ",docs[0].metadata)
print("content: ", docs[0].page_content[:1500])

C:\Users\Abhi\AppData\Local\Temp\ipykernel_17400\1215468004.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Documents loaded:  1
metadata:  {'source': 'https://docs.langchain.com/oss/python/langgraph/agentic-rag', 'title': 'Build a custom RAG agent with LangGraph - Docs by LangChain', 'description': 'Build a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.', 'language': 'en'}
content:  





Build a custom RAG agent with LangGraph - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchCo

#  Splitting the docs into chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
)

chunks = text_splitter.split_documents(docs)

print("Total chunks created: ", len(chunks))
print("First chunk:\n ",chunks[1].page_content[:900])

Total chunks created:  29
First chunk:
  Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchChunk documentsImprove retrieval qualitySQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. D


# Embedding vectors

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
    model_kwargs={"device": "cpu"},
)

sample_vector = embeddings.embed_query("Hello world")
print("Sample vector length: ", len(sample_vector))

c:\Developer\Enterprise AI assistant\assistant_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1210.33it/s]


Sample vector length:  384


# setting pinecone for vector DB

In [7]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import time

NAMESPACE = "IT-industry-agentic-rag"
INDEX_NAME = "agentic-rag"

# connect to pinecone
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"),
    environment="us-east1-gcp",
)

# create the index if it doesn't exist
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    
    while pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
        
print("pinecone index ready:", INDEX_NAME)

vector_store = PineconeVectorStore.from_documents(
    documents = chunks,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    embedding=embeddings,
)

base_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3,
        "namespace": NAMESPACE
        }
)

print("Pinecone vector store and retriever ready.")




pinecone index ready: agentic-rag
Pinecone vector store and retriever ready.


In [ ]:
# load existing Index

# loading the existing index from Pinecone
index = pc.Index(INDEX_NAME)

# connecting existing index to the vector store
vector_store = PineconeVectorStore(
    index=index,
    namespace=NAMESPACE,
    embedding=embeddings,
)